# Mixture of Experts (MoE)

## Learning Objectives

By the end of this notebook, you will:
- Understand **why** Mixture of Experts enables efficient scaling of neural networks
- Build a complete MoE layer from scratch with **sparse routing**
- Implement **gating mechanisms** that select which experts to use
- Learn about **load balancing** and why it's critical for training
- Visualize expert specialization and routing decisions
- Understand how MoE powers modern LLMs like Mixtral and GPT-4

## The Journey

We'll build understanding incrementally:

1. **Problem & Motivation** - Why do we need MoE?
2. **Dense vs Sparse Models** - The capacity-computation tradeoff
3. **Expert Networks** - Building specialized sub-networks
4. **Router Network** - Learning to select experts
5. **Sparse Gating** - Top-K selection for efficiency
6. **Load Balancing Loss** - Ensuring all experts are used
7. **Complete MoE Layer** - Putting it all together
8. **Training & Analysis** - Seeing expert specialization emerge
9. **Comparison** - MoE vs Dense models

**Our Task**: We'll train an MoE model on character-level language modeling and observe how different experts specialize in different patterns!

## Part 1: The Problem - Scaling Neural Networks

### The Capacity-Computation Dilemma

Larger models perform better, but there's a fundamental challenge:

**Dense Models**: Every input activates ALL parameters
- Want 10x capacity? → 10x parameters → 10x computation → 10x cost
- Linear scaling is expensive!

**Example**: A transformer feed-forward layer with 1B parameters:
- Every token processes through ALL 1B parameters
- Even simple inputs pay the full computational cost

### The MoE Solution: Conditional Computation

**Mixture of Experts**: Have many specialized "expert" networks, but only activate a few per input
- 8 experts, activate top-2 → 4x capacity, only 1.25x computation!
- **Sparse activation**: Different inputs use different experts
- **Specialization**: Experts learn to handle specific patterns

This is how models like **Mixtral 8x7B** achieve 47B parameter capacity with 13B active parameters per token!

## Part 2: Setup

Let's import our dependencies and set up the environment.

In [ ]:
# Core imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import seaborn as sns
from collections import defaultdict

# Shared utilities
from aiml_notebooks import (
    get_device,
    set_seed,
    create_dataset,
    create_dataloaders,
    count_parameters
)

# Enable autoreload
%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
set_seed(42)

# Device setup
device = get_device()

# Plotting configuration
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print("Setup complete!")

## Part 3: Dataset Preparation

We'll use character-level language modeling on names. This task is perfect for demonstrating MoE because:
- Different character patterns exist (vowels, consonants, endings, beginnings)
- Experts can specialize in different linguistic patterns
- Easy to visualize and understand

The model learns: given previous characters, predict the next character.

In [ ]:
# Load dataset
full_dataset, train_dataset, val_dataset = create_dataset(
    dataset_id="names",
    splits=[0.9, 0.1]
)

# Extract tokenizer
tokenizer = full_dataset.tokenizer
vocab_size = tokenizer.vocab_size

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(tokenizer.chars)}")
print(f"\nTraining examples: {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")

# Show examples
print("\nExample training pairs:")
for i in range(3):
    x, y = train_dataset[i]
    x_text = tokenizer.decode(x.tolist())
    y_text = tokenizer.decode(y.tolist())
    print(f"  Input:  '{x_text}' → Target: '{y_text}'")

## Part 4: Building Blocks - A Single Expert

### What is an Expert?

An **expert** is just a small neural network (usually a feed-forward network). In transformers, experts typically replace the FFN layer.

**Structure**: Input → Linear → ReLU → Linear → Output

Each expert has its own parameters and can learn to specialize in processing certain types of inputs.

In [ ]:
class Expert(nn.Module):
    """A single expert network (feed-forward)."""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Input/output dimension
            d_ff: Hidden dimension
            dropout: Dropout probability
        """
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model) or (num_tokens, d_model)
        Returns:
            (same shape as input)
        """
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

# Test a single expert
d_model = 64
d_ff = 256

expert = Expert(d_model, d_ff)
test_input = torch.randn(2, 10, d_model)  # (batch=2, seq_len=10, d_model=64)
test_output = expert(test_input)

print(f"Expert architecture:")
print(f"  Input dimension: {d_model}")
print(f"  Hidden dimension: {d_ff}")
print(f"  Output dimension: {d_model}")
print(f"  Parameters: {count_parameters(expert):,}")
print(f"\nTest:")
print(f"  Input shape:  {test_input.shape}")
print(f"  Output shape: {test_output.shape}")

## Part 5: Multiple Experts - Creating Diversity

Now let's create multiple experts. Each expert has the same architecture but **different parameters**, allowing them to specialize.

We'll create 8 experts (common choice in modern MoE models like Mixtral).

In [ ]:
# Create multiple experts
num_experts = 8

experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])

print(f"Created {num_experts} experts")
print(f"Parameters per expert: {count_parameters(experts[0]):,}")
print(f"Total parameters (all experts): {sum(count_parameters(e) for e in experts):,}")

# If we used ALL experts for every input (dense):
# Computation would be 8x a single expert
print(f"\nDense computation cost: {num_experts}x single expert")
print(f"MoE computation cost (top-2): 2x single expert")
print(f"Efficiency gain: {num_experts / 2:.1f}x more capacity for same compute!")

## Part 6: The Router Network - Learning to Choose

### The Core Question: Which experts should process this input?

The **router** (also called **gating network**) learns to select which experts are most suitable for each input.

**How it works**:
1. Input token → Router network → Scores for each expert
2. Apply softmax → Probability distribution over experts
3. Select top-K experts (usually K=1 or K=2)
4. Route input to selected experts

The router is learned end-to-end with the experts!

In [ ]:
class Router(nn.Module):
    """Router network that selects which experts to use."""
    
    def __init__(self, d_model, num_experts):
        """
        Args:
            d_model: Input dimension
            num_experts: Number of experts to choose from
        """
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts)
        self.num_experts = num_experts
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model) or (num_tokens, d_model)
        Returns:
            logits: (num_tokens, num_experts) - unnormalized scores
            probs: (num_tokens, num_experts) - probabilities (softmax)
        """
        # Flatten to (num_tokens, d_model) if needed
        original_shape = x.shape
        if x.dim() == 3:
            x = x.view(-1, x.size(-1))  # (batch * seq_len, d_model)
        
        # Compute expert scores
        logits = self.gate(x)  # (num_tokens, num_experts)
        probs = F.softmax(logits, dim=-1)  # Normalize to probabilities
        
        return logits, probs

# Test router
router = Router(d_model, num_experts)
test_input = torch.randn(2, 10, d_model)  # (batch=2, seq_len=10, d_model=64)
logits, probs = router(test_input)

print(f"Router:")
print(f"  Input: {d_model} dimensions")
print(f"  Output: {num_experts} expert scores")
print(f"  Parameters: {count_parameters(router):,}")
print(f"\nTest:")
print(f"  Input shape:  {test_input.shape}")
print(f"  Logits shape: {logits.shape}")
print(f"  Probs shape:  {probs.shape}")
print(f"\nExample routing probabilities for first token:")
print(f"  Expert scores: {probs[0].detach().numpy()}")
print(f"  Sum: {probs[0].sum():.4f} (should be 1.0)")

### Visualizing Router Decisions

Let's see how the router distributes probability across experts (initially random, since untrained).

In [ ]:
# Visualize routing for multiple tokens
num_tokens = 20
test_tokens = torch.randn(num_tokens, d_model)
_, routing_probs = router(test_tokens)

# Plot heatmap
plt.figure(figsize=(12, 8))
plt.imshow(routing_probs.detach().numpy(), aspect='auto', cmap='viridis')
plt.colorbar(label='Routing Probability')
plt.xlabel('Expert ID')
plt.ylabel('Token ID')
plt.title('Router Decisions (Untrained) - Which Expert for Each Token?')
plt.xticks(range(num_experts))
plt.tight_layout()
plt.show()

print("Reading the heatmap:")
print("- Each ROW is a token")
print("- Each COLUMN is an expert")
print("- Brighter = higher probability")
print("- Initially random (untrained router)")

## Part 7: Sparse Gating - Top-K Selection

### The Key Innovation: Sparsity

Instead of using ALL experts (expensive), we only use the **top-K** experts with highest routing probability.

**Common choices**:
- **Top-1**: Only the best expert (most efficient, but less capacity)
- **Top-2**: Best two experts (good balance - used in Mixtral)
- **Top-K**: More experts (more capacity, more computation)

**Benefits**:
- Computational efficiency (only activate K experts, not all)
- Sparse gradients (only selected experts get updated)
- Automatic specialization (experts compete to be selected)

In [ ]:
def top_k_gating(probs, k=2):
    """
    Select top-k experts and normalize their weights.
    
    Args:
        probs: (num_tokens, num_experts) - routing probabilities
        k: Number of experts to select
    
    Returns:
        gates: (num_tokens, num_experts) - sparse gating weights (most are 0)
        indices: (num_tokens, k) - indices of selected experts
    """
    # Get top-k values and indices
    top_k_probs, top_k_indices = torch.topk(probs, k, dim=-1)  # (num_tokens, k)
    
    # Renormalize top-k probabilities (so they sum to 1)
    top_k_gates = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
    
    # Create sparse gate tensor (mostly zeros)
    gates = torch.zeros_like(probs)  # (num_tokens, num_experts)
    gates.scatter_(1, top_k_indices, top_k_gates)  # Fill in top-k positions
    
    return gates, top_k_indices

# Test top-k gating
test_probs = torch.tensor([
    [0.1, 0.05, 0.3, 0.15, 0.2, 0.05, 0.1, 0.05],  # Token 1
    [0.4, 0.1, 0.1, 0.05, 0.05, 0.2, 0.05, 0.05],  # Token 2
])

print("Original probabilities:")
print(test_probs.numpy())
print(f"\nRow sums: {test_probs.sum(dim=1).numpy()} (all equal 1.0)")

gates, indices = top_k_gating(test_probs, k=2)

print("\nAfter top-2 gating:")
print(gates.numpy())
print(f"\nSelected experts for each token:")
print(f"  Token 1: Experts {indices[0].tolist()} with weights {gates[0][gates[0] > 0].tolist()}")
print(f"  Token 2: Experts {indices[1].tolist()} with weights {gates[1][gates[1] > 0].tolist()}")

print(f"\nSparsity: {(gates == 0).float().mean():.1%} of gates are zero")

### Visualizing Sparsity

Let's visualize the difference between dense (all experts) and sparse (top-k) gating.

In [ ]:
# Generate routing for multiple tokens
num_tokens = 20
test_tokens = torch.randn(num_tokens, d_model)
_, routing_probs = router(test_tokens)

# Apply top-2 gating
sparse_gates, selected_experts = top_k_gating(routing_probs, k=2)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Dense (all experts)
im1 = axes[0].imshow(routing_probs.detach().numpy(), aspect='auto', cmap='viridis', vmin=0, vmax=1)
axes[0].set_xlabel('Expert ID')
axes[0].set_ylabel('Token ID')
axes[0].set_title('Dense Gating (All Experts)')
plt.colorbar(im1, ax=axes[0], label='Weight')

# Sparse (top-2)
im2 = axes[1].imshow(sparse_gates.detach().numpy(), aspect='auto', cmap='viridis', vmin=0, vmax=1)
axes[1].set_xlabel('Expert ID')
axes[1].set_ylabel('Token ID')
axes[1].set_title('Sparse Gating (Top-2 Experts)')
plt.colorbar(im2, ax=axes[1], label='Weight')

plt.tight_layout()
plt.show()

print(f"Dense: All {num_experts} experts used per token")
print(f"Sparse (top-2): Only 2 experts used per token")
print(f"Computation reduction: {num_experts / 2:.1f}x fewer expert evaluations!")

## Part 8: Load Balancing Loss

### The Load Balancing Problem

**Problem**: Without constraints, the router might send all inputs to just 1-2 experts, leaving others unused!
- Wastes model capacity
- Prevents specialization
- Some experts never learn

**Solution**: Add a **load balancing loss** that encourages using all experts equally.

### Load Balancing Loss Formula

We want each expert to be selected approximately equally often:

$$L_{balance} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

Where:
- $f_i$ = fraction of tokens routed to expert $i$
- $P_i$ = average routing probability for expert $i$
- $N$ = number of experts
- $\alpha$ = balancing coefficient (typically 0.01)

This encourages $f_i \approx 1/N$ (equal usage).

In [ ]:
def load_balancing_loss(probs, gates, num_experts, alpha=0.01):
    """
    Compute load balancing loss to encourage equal expert usage.
    
    Args:
        probs: (num_tokens, num_experts) - routing probabilities from router
        gates: (num_tokens, num_experts) - sparse gating weights (top-k)
        num_experts: Number of experts
        alpha: Balancing coefficient
    
    Returns:
        loss: Scalar tensor
    """
    # Fraction of tokens assigned to each expert (based on gates)
    # Sum across tokens, divide by num_tokens
    f = gates.sum(dim=0) / gates.size(0)  # (num_experts,)
    
    # Average routing probability for each expert
    P = probs.mean(dim=0)  # (num_experts,)
    
    # Load balancing loss
    loss = alpha * num_experts * (f * P).sum()
    
    return loss, f, P

# Test load balancing loss
# Create imbalanced routing (most tokens go to expert 0)
imbalanced_probs = torch.zeros(100, num_experts)
imbalanced_probs[:, 0] = 0.8  # Expert 0 gets high probability
imbalanced_probs[:, 1:] = 0.2 / (num_experts - 1)  # Others share the rest

imbalanced_gates, _ = top_k_gating(imbalanced_probs, k=2)

lb_loss, frac, avg_prob = load_balancing_loss(
    imbalanced_probs, 
    imbalanced_gates, 
    num_experts
)

print("Imbalanced routing:")
print(f"  Fraction of tokens per expert: {frac.numpy()}")
print(f"  Average probability per expert: {avg_prob.numpy()}")
print(f"  Load balancing loss: {lb_loss.item():.6f}")

# Create balanced routing
balanced_probs = torch.ones(100, num_experts) / num_experts
balanced_gates, _ = top_k_gating(balanced_probs, k=2)

lb_loss_balanced, frac_balanced, avg_prob_balanced = load_balancing_loss(
    balanced_probs,
    balanced_gates,
    num_experts
)

print("\nBalanced routing:")
print(f"  Fraction of tokens per expert: {frac_balanced.numpy()}")
print(f"  Average probability per expert: {avg_prob_balanced.numpy()}")
print(f"  Load balancing loss: {lb_loss_balanced.item():.6f}")

print(f"\nLower loss = more balanced expert usage!")

### Visualizing Load Balance

Let's visualize what balanced vs imbalanced expert usage looks like.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Imbalanced
axes[0].bar(range(num_experts), frac.detach().numpy())
axes[0].axhline(y=1/num_experts, color='r', linestyle='--', label='Ideal (1/N)')
axes[0].set_xlabel('Expert ID')
axes[0].set_ylabel('Fraction of Tokens')
axes[0].set_title('Imbalanced Expert Usage')
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Balanced
axes[1].bar(range(num_experts), frac_balanced.detach().numpy())
axes[1].axhline(y=1/num_experts, color='r', linestyle='--', label='Ideal (1/N)')
axes[1].set_xlabel('Expert ID')
axes[1].set_ylabel('Fraction of Tokens')
axes[1].set_title('Balanced Expert Usage')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The load balancing loss pushes usage toward the red line (equal usage).")

## Part 9: Complete MoE Layer

Now let's combine everything into a complete Mixture of Experts layer:

1. **Router** computes expert probabilities
2. **Top-K gating** selects best experts
3. **Route tokens** to selected experts
4. **Combine outputs** weighted by gating values
5. **Load balancing loss** for training

In [ ]:
# Test MoE layer
moe = MoELayer(d_model, d_ff, num_experts, top_k=2)
test_input = torch.randn(2, 10, d_model)
test_output, test_lb_loss, routing_info = moe(test_input, return_routing_info=True)

print(f"MoE Layer:")
print(f"  {num_experts} experts, top-{moe.top_k} activated per token")
print(f"  Parameters: {count_parameters(moe):,}")
print(f"\nTest:")
print(f"  Input shape:  {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Load balance loss: {test_lb_loss.item():.6f}")
print(f"\nExpert usage:")
print(f"  {routing_info['expert_usage'].detach().numpy()}")

## Part 10: MoE vs Dense Comparison

Let's build a simple language model with both MoE and dense FFN layers to compare them.

We'll create:
1. **Dense model**: Standard transformer with regular FFN
2. **MoE model**: Transformer with MoE layer instead of FFN

Both will have similar **active** parameter counts for fair comparison.

In [ ]:
class SimpleLM(nn.Module):
    """Simple language model with either Dense or MoE layers."""
    
    def __init__(self, vocab_size, d_model, d_ff, num_layers, use_moe=False, 
                 num_experts=8, top_k=2, dropout=0.1):
        """
        Args:
            vocab_size: Size of vocabulary
            d_model: Model dimension
            d_ff: Feed-forward hidden dimension
            num_layers: Number of layers
            use_moe: If True, use MoE layers; else use dense FFN
            num_experts: Number of experts (if use_moe=True)
            top_k: Number of experts to activate (if use_moe=True)
            dropout: Dropout probability
        """
        super().__init__()
        self.use_moe = use_moe
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Layers
        if use_moe:
            self.layers = nn.ModuleList([
                MoELayer(d_model, d_ff, num_experts, top_k, dropout)
                for _ in range(num_layers)
            ])
        else:
            self.layers = nn.ModuleList([
                Expert(d_model, d_ff, dropout)
                for _ in range(num_layers)
            ])
        
        # Output
        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len) - token indices
        
        Returns:
            logits: (batch, seq_len, vocab_size)
            total_lb_loss: Load balancing loss (0 if not MoE)
        """
        # Embed
        x = self.embedding(x)  # (batch, seq_len, d_model)
        x = self.dropout(x)
        
        # Process through layers
        total_lb_loss = 0
        for layer in self.layers:
            if self.use_moe:
                x, lb_loss = layer(x)
                total_lb_loss += lb_loss
            else:
                x = layer(x)
        
        # Output
        x = self.norm(x)
        logits = self.output(x)
        
        return logits, total_lb_loss

# Create both models
num_layers = 2

dense_model = SimpleLM(
    vocab_size=vocab_size,
    d_model=d_model,
    d_ff=d_ff,
    num_layers=num_layers,
    use_moe=False
).to(device)

moe_model = SimpleLM(
    vocab_size=vocab_size,
    d_model=d_model,
    d_ff=d_ff,
    num_layers=num_layers,
    use_moe=True,
    num_experts=num_experts,
    top_k=2
).to(device)

print("Model Comparison:")
print(f"\nDense Model:")
print(f"  Total parameters: {count_parameters(dense_model):,}")
print(f"  Active parameters per token: {count_parameters(dense_model):,}")

# For MoE, active params = embedding + output + (router + top_k experts) per layer
moe_total = count_parameters(moe_model)
expert_params = count_parameters(moe_model.layers[0].experts[0])
router_params = count_parameters(moe_model.layers[0].router)
moe_active = count_parameters(moe_model.embedding) + count_parameters(moe_model.output) + \
             num_layers * (router_params + 2 * expert_params)  # top-2

print(f"\nMoE Model:")
print(f"  Total parameters: {moe_total:,}")
print(f"  Active parameters per token: ~{moe_active:,} (top-2 of {num_experts} experts)")
print(f"\nCapacity increase: {moe_total / count_parameters(dense_model):.1f}x")
print(f"Compute increase: {moe_active / count_parameters(dense_model):.2f}x")
print(f"Efficiency: {(moe_total / count_parameters(dense_model)) / (moe_active / count_parameters(dense_model)):.1f}x more capacity per FLOPs!")

## Part 11: Training Setup

Now let's train both models on the same task and compare:
- Training speed
- Final performance
- Expert specialization (for MoE)

We'll train on character-level language modeling.

In [ ]:
# Create data loaders
BATCH_SIZE = 128

train_loader, val_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0
)

print(f"Data:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

### Training Function

For MoE models, the total loss includes:
1. **Cross-entropy loss** (standard language modeling)
2. **Load balancing loss** (encourages equal expert usage)

Total loss = CE loss + load balancing loss

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device, is_moe=False):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_ce_loss = 0
    total_lb_loss = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        
        # Forward
        optimizer.zero_grad()
        logits, lb_loss = model(x)
        
        # Cross-entropy loss
        ce_loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        
        # Total loss
        loss = ce_loss + lb_loss
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        # Track
        total_loss += loss.item()
        total_ce_loss += ce_loss.item()
        total_lb_loss += lb_loss.item() if is_moe else 0
        
        if is_moe:
            pbar.set_postfix({
                'loss': f'{loss.item():.3f}',
                'ce': f'{ce_loss.item():.3f}',
                'lb': f'{lb_loss.item():.4f}'
            })
        else:
            pbar.set_postfix({'loss': f'{loss.item():.3f}'})
    
    n = len(train_loader)
    return total_loss / n, total_ce_loss / n, total_lb_loss / n

def evaluate(model, val_loader, criterion, device, is_moe=False):
    """Evaluate on validation set."""
    model.eval()
    total_loss = 0
    total_ce_loss = 0
    
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, lb_loss = model(x)
            ce_loss = criterion(logits.view(-1, vocab_size), y.view(-1))
            loss = ce_loss + lb_loss
            total_loss += loss.item()
            total_ce_loss += ce_loss.item()
    
    n = len(val_loader)
    return total_loss / n, total_ce_loss / n

print("Training functions defined!")

### Train Both Models

Let's train both the dense and MoE models and compare their performance.

In [ ]:
# Training config
learning_rate = 3e-4
num_epochs = 5
criterion = nn.CrossEntropyLoss()

# Optimizers
dense_optimizer = torch.optim.AdamW(dense_model.parameters(), lr=learning_rate)
moe_optimizer = torch.optim.AdamW(moe_model.parameters(), lr=learning_rate)

# Track history
dense_history = {'train_loss': [], 'val_loss': []}
moe_history = {'train_loss': [], 'val_loss': [], 'train_ce': [], 'train_lb': []}

print("Training configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"\nStarting training...\n")

In [ ]:
# Train Dense Model
print("="*50)
print("TRAINING DENSE MODEL")
print("="*50)

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    train_loss, _, _ = train_epoch(dense_model, train_loader, dense_optimizer, criterion, device, is_moe=False)
    val_loss, _ = evaluate(dense_model, val_loader, criterion, device, is_moe=False)
    
    dense_history['train_loss'].append(train_loss)
    dense_history['val_loss'].append(val_loss)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("\n✓ Dense model training complete!")

In [ ]:
# Train MoE Model
print("\n" + "="*50)
print("TRAINING MoE MODEL")
print("="*50)

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    train_loss, train_ce, train_lb = train_epoch(moe_model, train_loader, moe_optimizer, criterion, device, is_moe=True)
    val_loss, val_ce = evaluate(moe_model, val_loader, criterion, device, is_moe=True)
    
    moe_history['train_loss'].append(train_loss)
    moe_history['val_loss'].append(val_loss)
    moe_history['train_ce'].append(train_ce)
    moe_history['train_lb'].append(train_lb)
    
    print(f"Train Loss: {train_loss:.4f} (CE: {train_ce:.4f}, LB: {train_lb:.4f}) | Val Loss: {val_loss:.4f}")

print("\n✓ MoE model training complete!")

## Part 12: Comparing Performance

Let's visualize the training curves and compare the models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, num_epochs + 1)

# Training loss
axes[0].plot(epochs, dense_history['train_loss'], marker='o', label='Dense', linewidth=2)
axes[0].plot(epochs, moe_history['train_loss'], marker='s', label='MoE', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation loss
axes[1].plot(epochs, dense_history['val_loss'], marker='o', label='Dense', linewidth=2)
axes[1].plot(epochs, moe_history['val_loss'], marker='s', label='MoE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal Results:")
print(f"Dense Model:  Val Loss = {dense_history['val_loss'][-1]:.4f}")
print(f"MoE Model:    Val Loss = {moe_history['val_loss'][-1]:.4f}")

if moe_history['val_loss'][-1] < dense_history['val_loss'][-1]:
    improvement = (dense_history['val_loss'][-1] - moe_history['val_loss'][-1]) / dense_history['val_loss'][-1] * 100
    print(f"\nMoE is {improvement:.1f}% better despite similar active parameters!")
else:
    print(f"\nDense model performs slightly better (may need more training for MoE to shine)")

### MoE Loss Components

For the MoE model, let's see how the load balancing loss evolved during training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CE loss over time
axes[0].plot(epochs, moe_history['train_ce'], marker='o', label='CE Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('MoE: Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Load balancing loss over time
axes[1].plot(epochs, moe_history['train_lb'], marker='s', label='Load Balance Loss', color='orange', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('MoE: Load Balancing Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The load balancing loss ensures all experts are utilized during training.")

## Part 13: Analyzing Expert Specialization

Now for the most exciting part: **What did the experts learn to do?**

Let's analyze:
1. Which experts are used most often?
2. Do different experts handle different types of inputs?
3. Are there patterns in expert selection?

In [ ]:
def analyze_expert_usage(model, dataloader, device, num_batches=10):
    """
    Analyze which experts are used for different inputs.
    
    Returns:
        expert_counts: (num_experts,) - how often each expert is selected
        char_to_expert: dict mapping characters to their most common expert
        routing_matrix: (num_tokens, num_experts) - routing probabilities
    """
    model.eval()
    
    expert_counts = torch.zeros(num_experts)
    all_routing_probs = []
    all_chars = []
    
    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(dataloader):
            if batch_idx >= num_batches:
                break
            
            x = x.to(device)
            batch_size, seq_len = x.shape
            
            # Get embeddings
            embeddings = model.embedding(x)  # (batch, seq_len, d_model)
            
            # Get routing from first MoE layer
            moe_layer = model.layers[0]
            embeddings_flat = embeddings.view(-1, d_model)
            _, probs = moe_layer.router(embeddings_flat)
            
            # Get top-k indices
            _, top_k_indices = top_k_gating(probs, k=2)
            
            # Count expert usage
            for expert_idx in range(num_experts):
                expert_counts[expert_idx] += (top_k_indices == expert_idx).sum().item()
            
            # Store routing probs and chars
            all_routing_probs.append(probs.cpu())
            all_chars.extend(x.view(-1).cpu().tolist())
    
    # Combine routing probs
    all_routing_probs = torch.cat(all_routing_probs, dim=0)
    
    # Analyze character to expert mapping
    char_to_expert = {}
    for char_idx in range(vocab_size):
        # Find all positions with this character
        char_mask = torch.tensor([c == char_idx for c in all_chars])
        if char_mask.sum() > 0:
            # Average routing for this character
            char_routing = all_routing_probs[char_mask].mean(dim=0)
            # Most preferred expert
            preferred_expert = char_routing.argmax().item()
            char_to_expert[char_idx] = preferred_expert
    
    return expert_counts, char_to_expert, all_routing_probs

# Analyze MoE model
expert_counts, char_to_expert, routing_probs = analyze_expert_usage(
    moe_model, val_loader, device, num_batches=20
)

print("Expert Usage Statistics:")
print(f"\nTotal selections per expert:")
for i, count in enumerate(expert_counts):
    print(f"  Expert {i}: {int(count):,} selections")

# Check balance
total_selections = expert_counts.sum()
expected_per_expert = total_selections / num_experts
print(f"\nExpected per expert (balanced): {expected_per_expert:.0f}")
print(f"Actual range: {expert_counts.min():.0f} - {expert_counts.max():.0f}")

# Most/least used
most_used = expert_counts.argmax().item()
least_used = expert_counts.argmin().item()
print(f"\nMost used expert: {most_used} ({int(expert_counts[most_used])} selections)")
print(f"Least used expert: {least_used} ({int(expert_counts[least_used])} selections)")

### Visualizing Expert Usage

Let's see the distribution of expert usage.

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(range(num_experts), expert_counts.numpy())
plt.axhline(y=expert_counts.mean(), color='r', linestyle='--', label=f'Average ({expert_counts.mean():.0f})')
plt.xlabel('Expert ID')
plt.ylabel('Number of Selections')
plt.title('Expert Usage Distribution (Trained MoE Model)')
plt.xticks(range(num_experts))
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Calculate imbalance
imbalance = (expert_counts.std() / expert_counts.mean()).item()
print(f"\nExpert usage imbalance (lower is better): {imbalance:.3f}")
print(f"Perfect balance would be: 0.000")
print(f"\nThe load balancing loss keeps experts reasonably balanced!")

### Character-to-Expert Mapping

Do different characters prefer different experts? This would indicate specialization!

In [ ]:
# Create character to expert preference matrix
char_expert_matrix = torch.zeros(vocab_size, num_experts)

for char_idx in range(vocab_size):
    if char_idx in char_to_expert:
        # Find all positions with this character
        char_positions = [i for i, c in enumerate(all_chars) if c == char_idx]
        if char_positions:
            char_routing = routing_probs[char_positions].mean(dim=0)
            char_expert_matrix[char_idx] = char_routing

# Plot heatmap
plt.figure(figsize=(12, 8))
plt.imshow(char_expert_matrix.numpy(), aspect='auto', cmap='viridis')
plt.colorbar(label='Average Routing Probability')
plt.xlabel('Expert ID')
plt.ylabel('Character')
plt.title('Character-to-Expert Routing Preferences')
plt.yticks(range(vocab_size), tokenizer.chars)
plt.xticks(range(num_experts))
plt.tight_layout()
plt.show()

print("Reading the heatmap:")
print("- Each ROW is a character")
print("- Each COLUMN is an expert")
print("- Brighter = this expert is more often selected for this character")
print("\nLook for patterns:")
print("- Do vowels prefer certain experts?")
print("- Do consonants cluster to different experts?")
print("- Is there visible specialization?")

### Expert Specialization Analysis

Let's group characters by their preferred expert and see if patterns emerge.

In [ ]:
# Group characters by preferred expert
expert_to_chars = {i: [] for i in range(num_experts)}

for char_idx, expert_idx in char_to_expert.items():
    char = tokenizer.decode_char(char_idx)
    expert_to_chars[expert_idx].append(char)

print("Expert Specialization - Which characters does each expert handle?\n")
print("="*60)

for expert_idx in range(num_experts):
    chars = expert_to_chars[expert_idx]
    if chars:
        print(f"Expert {expert_idx}: {' '.join(sorted(chars))}")
        
        # Check for patterns
        vowels = set('aeiou')
        char_set = set(chars)
        vowel_chars = char_set & vowels
        consonant_chars = char_set - vowels - {'.'}
        
        if vowel_chars:
            print(f"  → Contains vowels: {' '.join(sorted(vowel_chars))}")
        if consonant_chars:
            print(f"  → Contains consonants: {' '.join(sorted(consonant_chars))}")
        if '.' in chars:
            print(f"  → Handles special token: .")
    else:
        print(f"Expert {expert_idx}: (no clear preference)")
    print()

print("="*60)
print("\nNotice any patterns? Experts may specialize in:")
print("  - Vowels vs consonants")
print("  - Common vs rare characters")
print("  - Beginning vs ending characters")
print("  - Specific phonetic patterns")

## Part 14: Generating Text with MoE

Let's generate some names using the MoE model and trace which experts are being used!

In [ ]:
def generate_with_expert_tracking(model, tokenizer, start_char, max_length=15, temperature=1.0):
    """
    Generate text and track which experts are used.
    
    Returns:
        text: Generated string
        expert_sequence: List of expert IDs used at each step
    """
    model.eval()
    
    # Start with special token
    idx = torch.tensor([[tokenizer.get_special_token_idx()]], device=device)
    expert_sequence = []
    
    with torch.no_grad():
        for _ in range(max_length):
            # Get embeddings
            x = model.embedding(idx)
            
            # Get routing from first MoE layer (for last token)
            moe_layer = model.layers[0]
            last_token_emb = x[:, -1:, :]  # (1, 1, d_model)
            _, probs = moe_layer.router(last_token_emb)
            top_expert = probs.argmax().item()
            expert_sequence.append(top_expert)
            
            # Generate next token
            logits, _ = model(idx)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
            
            # Stop at special token
            if next_idx.item() == tokenizer.get_special_token_idx():
                break
    
    # Decode
    text = tokenizer.decode(idx[0].cpu().tolist())
    special = tokenizer.special_token
    if text.count(special) >= 2:
        text = text.split(special)[1]
    else:
        text = text.replace(special, '')
    
    return text, expert_sequence

# Generate some names
print("Generated Names with Expert Tracking:\n")
print("="*60)

for i in range(10):
    name, experts = generate_with_expert_tracking(moe_model, tokenizer, '.', max_length=15, temperature=0.8)
    expert_str = ' → '.join([f"E{e}" for e in experts[:len(name)]])
    print(f"{i+1:2d}. {name:15s} | Experts: {expert_str}")

print("="*60)
print("\nNotice:")
print("  - Different positions may use different experts")
print("  - Some experts may be used more frequently")
print("  - This is conditional computation in action!")

## Part 15: Key Takeaways

### What We Learned

Congratulations! You've built a complete Mixture of Experts system from scratch. Let's recap the key insights:

#### 1. **The Core Innovation: Conditional Computation**
- Dense models activate ALL parameters for every input (expensive)
- MoE models activate only top-K experts per input (efficient)
- Result: More capacity without proportional increase in computation

#### 2. **Router Network: The Brain of MoE**
- Learns to select which experts are most suitable for each input
- Trained end-to-end with experts via backpropagation
- Enables dynamic, input-dependent computation

#### 3. **Sparse Gating: Top-K Selection**
- Only activate K out of N experts (typically K=1 or K=2)
- Provides huge efficiency gains (8 experts, use 2 → 4x capacity, 1.25x cost)
- Makes large models computationally feasible

#### 4. **Load Balancing: Ensuring All Experts Learn**
- Without constraints, router may collapse to using few experts
- Load balancing loss encourages equal expert usage
- Critical for training stable MoE models

#### 5. **Expert Specialization: Emergent Behavior**
- Experts automatically specialize in different patterns
- May cluster by: character types, linguistic patterns, frequency, etc.
- Specialization emerges from training, not hand-coding!

### MoE in Modern LLMs

The architecture you built is the same one used in:
- **Mixtral 8x7B**: 8 experts, top-2 routing, 47B total params, 13B active
- **GPT-4**: Rumored to use MoE (16 experts)
- **Switch Transformer**: Up to 1.6 trillion parameters with MoE

### The Scaling Recipe

MoE enables a new scaling law:
1. Want more capacity? Add more experts!
2. Keep compute fixed by using sparse top-K
3. Each expert specializes
4. Overall model becomes much more capable

**Result**: Models can grow to trillions of parameters while remaining trainable and efficient!

## Part 16: Further Exploration

### Experiments to Try

Deepen your understanding by experimenting:

#### 1. **Architecture Changes**
- **More experts**: Try 16 or 32 experts
  - Does specialization become more pronounced?
  - How does load balancing change?

- **Different top-K**: Try top-1 or top-4
  - Top-1: Most efficient, but less capacity
  - Top-4: More capacity, but higher cost

- **Deeper models**: Add more layers
  - Do different layers learn different specializations?

#### 2. **Training Changes**
- **Load balancing coefficient**: Try α = 0.001 or α = 0.1
  - Too low → experts collapse
  - Too high → forces unnatural balance, hurts performance

- **No load balancing**: Set α = 0
  - Watch experts collapse to just 1-2
  - Demonstrates why load balancing is critical

#### 3. **Analysis**
- **Expert similarity**: Compute parameter similarity between experts
  - Do experts remain different?
  - Or do they converge?

- **Per-expert performance**: Evaluate each expert individually
  - Which expert is "best"?
  - Are all experts contributing?

#### 4. **Alternative Gating**
- **Noisy top-K**: Add noise to routing logits during training
  - Helps with exploration
  - Used in original MoE paper

- **Expert choice routing**: Let experts choose which tokens to process
  - Reverses the routing direction!
  - Better load balancing

Try these experiments below!

In [ ]:
# Experiment space - try modifications here!

# Example: Visualize routing entropy (how "decisive" is the router?)
def compute_routing_entropy(probs):
    """Compute entropy of routing distribution (higher = more uncertain)."""
    # Avoid log(0)
    probs = torch.clamp(probs, min=1e-10)
    entropy = -(probs * torch.log(probs)).sum(dim=-1)
    return entropy

# Compute entropy for all routing decisions
routing_entropy = compute_routing_entropy(routing_probs)

plt.figure(figsize=(10, 6))
plt.hist(routing_entropy.numpy(), bins=50, edgecolor='black')
plt.xlabel('Routing Entropy')
plt.ylabel('Frequency')
plt.title('Distribution of Routing Entropy')
plt.axvline(x=np.log(num_experts), color='r', linestyle='--', label=f'Max entropy (log({num_experts}))')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean routing entropy: {routing_entropy.mean():.3f}")
print(f"Max possible entropy: {np.log(num_experts):.3f} (uniform distribution)")
print(f"\nLower entropy = more decisive routing (good!)")
print(f"Higher entropy = uncertain routing (may indicate issues)")

## Final Reflections

### Why MoE Matters

Mixture of Experts represents a fundamental shift in how we scale neural networks:

**Before MoE**:
- Scaling = Adding more parameters = More computation
- Every input pays full computational cost
- Models became prohibitively expensive

**With MoE**:
- Scaling = Adding more experts (capacity) ≠ Adding computation
- Each input uses only relevant experts (sparse)
- Models can grow much larger efficiently

### The Future of MoE

Current trends and research directions:

1. **Scaling**: Models with hundreds or thousands of experts
2. **Granularity**: Expert per layer, per attention head, even per neuron
3. **Dynamic routing**: Learned routing strategies beyond softmax
4. **Multi-modal experts**: Specialized experts for text, images, audio, etc.
5. **Efficient training**: Better load balancing, expert initialization, distributed training

### Connection to Other Concepts

MoE relates to several important ML concepts:

- **Ensemble methods**: Multiple models, but learned jointly
- **Attention**: Similar "routing" mechanism (attention is soft routing!)
- **Neural architecture search**: Conditional computation paths
- **Modular networks**: Specialized sub-networks

### Key Papers to Read

1. "Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer" (Shazeer et al., 2017)
2. "Switch Transformers: Scaling to Trillion Parameter Models" (Fedus et al., 2021)
3. "Mixtral of Experts" (Jiang et al., 2024)
4. "ST-MoE: Designing Stable and Transferable Sparse Expert Models" (Zoph et al., 2022)

**Congratulations on mastering Mixture of Experts!** 🎉

You now understand one of the key technologies enabling the next generation of large language models!